# 🔬 PyCupFM — Panel Cointegration with Common Factors

**Author:** Dr. Merwan Roudane ([merwanroudane920@gmail.com](mailto:merwanroudane920@gmail.com))  
**GitHub:** [github.com/merwanroudane/cupfm](https://github.com/merwanroudane/cupfm)  
**Version:** 1.0.0

---

This notebook demonstrates the full capabilities of `pycupfm`, the Python implementation of all 5 panel cointegration estimators from:

- **Bai, J., Kao, C. & Ng, S. (2009)**. *Panel cointegration with global stochastic trends*. Journal of Econometrics, 149(1), 82-99.
- **Bai, J. & Kao, C. (2005)**. *On the estimation and inference of a panel cointegration model with cross-sectional dependence*. SSRN-1815227.

### Estimators Implemented

| Estimator | Method | Recommended |
|-----------|--------|:---:|
| LSDV | Within/Fixed-effects | ❌ (biased) |
| Bai FM | Two-step Fully Modified | |
| **CupFM** | Continuously-Updated FM | ⭐ |
| CupFM-bar | CupFM with Z-bar | |
| CupBC | Continuously-Updated BC | |

In [ ]:
# Install if needed
# !pip install pycupfm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pycupfm import CupFM, simulate_panel, monte_carlo
from pycupfm.datasets import load_grunfeld
from pycupfm.plotting import (
    plot_coefficients, plot_factors, plot_loadings,
    plot_convergence, plot_omega_heatmap, plot_factor_ic,
    plot_loadings_heatmap
)
from pycupfm.factors import bai_ng_ic

print('PyCupFM loaded successfully!')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')

---
## 1. 📊 Real Data Application — Grunfeld Investment Dataset

The classic Grunfeld (1958) dataset: **N = 10 firms, T = 20 years (1935–1954)**.

Model: $\ln(\text{invest})_{it} = \alpha_i + \beta_1 \ln(\text{mvalue})_{it} + \beta_2 \ln(\text{kstock})_{it} + \lambda_i' F_t + u_{it}$

In [ ]:
# Load the Grunfeld dataset
df = load_grunfeld()
print(f'Dataset: {len(df)} observations')
print(f'Firms: {df["firm"].nunique()}, Years: {df["year"].nunique()}')
print(f'Variables: {list(df.columns)}')
df.head(10)

In [ ]:
# Descriptive statistics
print('\n=== Descriptive Statistics ===')
df[['linvest', 'lmvalue', 'lkstock']].describe().round(4)

### 1.1 Fit CupFM with 1 Common Factor

In [ ]:
# Create the CupFM model
model = CupFM(
    n_factors=1,          # 1 common factor
    bandwidth=3,          # Bartlett kernel bandwidth (suitable for T=20)
    kernel='bartlett',    # Bartlett kernel
    max_iter=25,          # Max CupFM/CupBC iterations
)

# Fit all 5 estimators
results = model.fit(
    y=df['linvest'],
    X=df[['lmvalue', 'lkstock']],
    panel_id=df['firm'],
    time_id=df['year'],
    var_names=['lmvalue', 'lkstock'],
    dep_var='linvest'
)

# Print publication-quality summary
results.summary()

### 1.2 Results as DataFrame

In [ ]:
# Convert to pandas DataFrame for further analysis
results_df = results.to_dataframe()
results_df.round(4)

---
## 2. 🎨 Publication-Quality Visualizations

### 2.1 Coefficient Comparison Forest Plot

In [ ]:
fig = plot_coefficients(results, figsize=(12, 5))
plt.show()

### 2.2 Estimated Common Factors

In [ ]:
fig = plot_factors(results, figsize=(12, 5))
plt.show()

### 2.3 Factor Loadings

In [ ]:
fig = plot_loadings(results, figsize=(10, 5))
plt.show()

### 2.4 CupFM Convergence Path

In [ ]:
fig = plot_convergence(results, figsize=(10, 5))
plt.show()

### 2.5 Long-Run Covariance Ω Heatmap

In [ ]:
fig = plot_omega_heatmap(results, figsize=(7, 6))
plt.show()

### 2.6 Factor Loadings Heatmap

In [ ]:
fig = plot_loadings_heatmap(results, figsize=(6, 6))
plt.show()

---
## 3. 🔍 Automatic Factor Selection (Bai & Ng 2002)

In [ ]:
# Auto-select number of factors
model_auto = CupFM(n_factors='auto', bandwidth=3, auto_rmax=5, verbose=True)
results_auto = model_auto.fit(
    y=df['linvest'],
    X=df[['lmvalue', 'lkstock']],
    panel_id=df['firm'],
    time_id=df['year'],
    var_names=['lmvalue', 'lkstock'],
    dep_var='linvest'
)
print(f'\nAuto-selected r = {results_auto.r}')
results_auto.summary()

In [ ]:
# Plot IC values
if model_auto.ic_values_ is not None:
    fig = plot_factor_ic(model_auto.ic_values_, figsize=(8, 5))
    plt.show()

---
## 4. 🧪 Monte Carlo Simulation (BKN 2009 DGP)

Replicate the simulation design from Bai, Kao & Ng (2009) Tables 1-4:
- $N = 20$, $T = 40$, $k = 1$, $r = 2$
- True $\beta = 2.0$
- AR(1) errors with $\rho = 0.3$
- Correlated $(u, v)$ innovations

In [ ]:
# Simulate a single panel
sim = simulate_panel(N=20, T=40, k=1, r=2, beta=2.0, seed=42)
print(f'Simulated panel: N={20}, T={40}, true beta={2.0}')
print(f'Data shape: {sim["data"].shape}')
sim['data'].head()

In [ ]:
# Fit CupFM on simulated data
model_sim = CupFM(n_factors=2, bandwidth=5, max_iter=20)
results_sim = model_sim.fit(
    y=sim['data']['y'],
    X=sim['data'][['x1']],
    panel_id=sim['data']['panel_id'],
    time_id=sim['data']['time_id'],
    var_names=['x1'],
    dep_var='y'
)
results_sim.summary()

print(f'\n  True beta  = {sim["beta_true"][0]:.4f}')
print(f'  CupFM bias = {results_sim.beta[0] - sim["beta_true"][0]:.4f}')
print(f'  LSDV  bias = {results_sim.betas["LSDV"][0] - sim["beta_true"][0]:.4f}')

In [ ]:
# Visualize simulated data results
fig = plot_coefficients(results_sim, figsize=(8, 5))
plt.show()

In [ ]:
fig = plot_factors(results_sim, figsize=(12, 5))
plt.show()

In [ ]:
fig = plot_convergence(results_sim, figsize=(10, 5))
plt.show()

### 4.1 Monte Carlo Experiment (50 replications)

In [ ]:
# Run Monte Carlo (use n_reps=500+ for publication; 50 for demo speed)
mc_results = monte_carlo(
    n_reps=50, N=20, T=40, k=1, r=2,
    beta=2.0, bandwidth=5, max_iter=20,
    seed=42, verbose=True
)

In [ ]:
# Summary statistics
mc_summary = mc_results.groupby('estimator').agg(
    Mean_Beta=('beta_hat', 'mean'),
    Mean_Bias=('bias', 'mean'),
    Std_Bias=('bias', 'std'),
    RMSE=('bias', lambda x: np.sqrt((x**2).mean())),
    N_reps=('rep', 'nunique')
).round(4)

# Reorder to match BKN paper
order = ['LSDV', 'Bai FM', 'CupFM', 'CupFM-bar', 'CupBC']
mc_summary = mc_summary.loc[[e for e in order if e in mc_summary.index]]
print('\n=== Monte Carlo Results (BKN 2009 DGP) ===')
print(f'True β = 2.0, N=20, T=40, r=2, {mc_results["rep"].nunique()} reps\n')
mc_summary

In [ ]:
# Beautiful MC bias boxplot
import seaborn as sns
from pycupfm.plotting import _setup_style, COLORS

_setup_style()
fig, ax = plt.subplots(figsize=(10, 6))

palette = [COLORS.get(e, '#999') for e in order if e in mc_results['estimator'].unique()]
sns.boxplot(
    data=mc_results, x='estimator', y='bias',
    order=[e for e in order if e in mc_results['estimator'].unique()],
    palette=palette, width=0.6, linewidth=1.5,
    flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5},
    ax=ax
)
ax.axhline(y=0, color='#E74C3C', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Estimator', fontsize=13)
ax.set_ylabel('Bias (β̂ - β)', fontsize=13)
ax.set_title('Monte Carlo Bias Distribution — BKN (2009) DGP',
             fontsize=15, fontweight='bold')
fig.text(0.5, -0.02, f'N=20, T=40, r=2, β=2.0, {mc_results["rep"].nunique()} replications',
         ha='center', fontsize=10, color='#7F8C8D')
plt.tight_layout()
plt.show()

---
## 5. 🔧 Bandwidth Sensitivity Analysis

In [ ]:
# Test robustness across bandwidth values
bw_values = [2, 3, 4, 5, 6, 8]
bw_results = []

for bw in bw_values:
    m = CupFM(n_factors=1, bandwidth=bw, max_iter=25)
    r = m.fit(y=df['linvest'], X=df[['lmvalue', 'lkstock']],
              panel_id=df['firm'], time_id=df['year'],
              var_names=['lmvalue', 'lkstock'])
    for j, vn in enumerate(['lmvalue', 'lkstock']):
        bw_results.append({
            'Bandwidth': bw,
            'Variable': vn,
            'CupFM': r.betas['CupFM'][j],
            'CupBC': r.betas['CupBC'][j],
            'Bai FM': r.betas['Bai FM'][j],
        })

bw_df = pd.DataFrame(bw_results)
print('\n=== Bandwidth Sensitivity ===')
bw_df.round(4)

In [ ]:
# Bandwidth sensitivity plot
_setup_style()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, var in enumerate(['lmvalue', 'lkstock']):
    ax = axes[idx]
    subset = bw_df[bw_df['Variable'] == var]
    for est, color in [('CupFM', '#27AE60'), ('CupBC', '#C0392B'), ('Bai FM', '#E67E22')]:
        ax.plot(subset['Bandwidth'], subset[est], marker='o', linewidth=2,
                color=color, label=est, markersize=7)
    ax.set_xlabel('Bandwidth (M)', fontsize=12)
    ax.set_ylabel('Coefficient', fontsize=12)
    ax.set_title(f'{var}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)

fig.suptitle('Bandwidth Sensitivity Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. 📤 Export Results

In [ ]:
# LaTeX table
latex = results.to_latex(caption='CupFM Panel Cointegration — Grunfeld Data')
print(latex)

In [ ]:
# Export to all formats
from pycupfm import export_results
export_results(results, 'cupfm_grunfeld', fmt='all')
print('Exported: cupfm_grunfeld.csv, .xlsx, .tex, .html')

---
## 7. 📚 Summary

This notebook demonstrated:

1. **Real data application** with the Grunfeld dataset (N=10, T=20)
2. **All 5 estimators** (LSDV, Bai FM, CupFM, CupFM-bar, CupBC)
3. **6 visualization types** (coefficients, factors, loadings, convergence, Ω heatmap, loadings heatmap)
4. **Automatic factor selection** via Bai-Ng (2002) IC
5. **Monte Carlo simulation** replicating BKN (2009) DGP
6. **Bandwidth sensitivity** robustness check
7. **Export** to LaTeX, Excel, CSV, and HTML

### References

- Bai, J., Kao, C. & Ng, S. (2009). *Panel cointegration with global stochastic trends*. JoE 149(1), 82-99.
- Bai, J. & Kao, C. (2005). SSRN-1815227.
- Bai, J. & Ng, S. (2002). *Econometrica* 70(1), 191-221.

**Author:** Dr. Merwan Roudane — [merwanroudane920@gmail.com](mailto:merwanroudane920@gmail.com)